In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# ml_scan — Colab training notebook

Colab-GPU equivalent of `user_command.ipynb`. **No TimescaleDB.** Training bars come from
`train_ohlcv_60minute.parquet` on Google Drive (`MyDrive/ml_train`).

**Colab setup (once)**
1. Runtime → Change runtime type → **T4 GPU** (or A100).
2. Upload `train_ohlcv_60minute.parquet` (from `scan-trade export-train-parquet`) to Drive folder `ml_train`.
3. File → Open notebook from GitHub → `VipiChan/ml_scanner` → `colab_commands.ipynb`, **or** run the clone cell below.
4. Run cells in order. After `pip install`, if imports fail: Runtime → Restart session, then re-run from the **paths** cell.

**What this notebook does not do**
- It does not query the warehouse. CLI commands `features hourly`, `features mtf`, `scan`, `backtest run`, and `e2e` still expect Timescale — do not use them here.
- 15-minute features are included only if you also upload `train_ohlcv_15minute.parquet`. Hourly bars cannot reconstruct 15m honestly; missing `m15_*` columns are dropped at QC.
- Daily context is resampled from hourly (including the 15:15 IST close print) unless you upload `train_ohlcv_day.parquet`.



## 0 — Clone repo, install package, mount Drive



In [4]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Define your base Drive path
    BASE_PATH = Path("/content/drive/MyDrive/ml_train")
    os.chdir(BASE_PATH)

    # Clone the repo if it's not present
    if not (BASE_PATH / "ml_scanner").exists() and not (BASE_PATH / "src").exists():
        print("Source code not found. Cloning public repository...")
        !git clone https://github.com/VipiChan/ml_scanner.git

    # 1. Find the project root (where setup.py or src folder exists)
    REPO_DIR = BASE_PATH
    src_candidates = list(BASE_PATH.rglob("src"))

    if src_candidates:
        # Set REPO_DIR to the folder containing the 'src' folder
        REPO_DIR = src_candidates[0].parent
        print(f"Project root identified at: {REPO_DIR}")

    os.chdir(REPO_DIR)

    # 2. Add the 'src' folder to sys.path so 'import ml_scan' works
    src_path = str(REPO_DIR / "src")
    if os.path.isdir(src_path):
        if src_path not in sys.path:
            sys.path.insert(0, src_path)
        print(f"Successfully added to sys.path: {src_path}")
    else:
        # If 'src' isn't there, maybe the modules are in the root
        if str(REPO_DIR) not in sys.path:
            sys.path.insert(0, str(REPO_DIR))
        print(f"'src' not found. Adding root to sys.path: {REPO_DIR}")

    # 3. Optional: Install the project in editable mode if a setup file exists
    #if (REPO_DIR / "setup.py").exists() or (REPO_DIR / "pyproject.toml").exists():
        #!pip install -q -e "."

print("IN_COLAB:", IN_COLAB)
print("Current Working Directory:", Path.cwd())

Project root identified at: /content/drive/MyDrive/ml_train/ml_scanner
Successfully added to sys.path: /content/drive/MyDrive/ml_train/ml_scanner/src
IN_COLAB: True
Current Working Directory: /content/drive/MyDrive/ml_train/ml_scanner


## 0b — Paths, Drive parquet, settings

Edit `DRIVE_DIR` if your folder is not `MyDrive/ml_train`.
Set `SYMBOL_LIMIT` to an int (e.g. `20`) for a cheap smoke pass. `None` = every symbol in the parquet.



In [6]:
import os
import sys
from pathlib import Path
import pandas as pd

# Use the REPO_DIR established in the previous cell
if 'REPO_DIR' not in globals():
    REPO_DIR = Path("/content/drive/MyDrive/ml_train")

os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from ml_scan.config import load_settings
from ml_scan.data.parquet_source import load_training_bundle
from ml_scan.ml_engine.estimator import describe_accelerator, gpu_available

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.expand_frame_repr", False)

# --- Paths configuration ---
DRIVE_DIR = REPO_DIR
OHLCV_PATH = DRIVE_DIR / "train_ohlcv_60minute.parquet"
ARTIFACTS = DRIVE_DIR / "artifacts"
SYMBOL_LIMIT = None
SAMPLE_ROWS = 300_000
# --------------------------

ARTIFACTS.mkdir(parents=True, exist_ok=True)
assert OHLCV_PATH.is_file(), f"File not found at {OHLCV_PATH}. Please verify the file is in your ml_train folder."

settings = load_settings()
hourly, bench_daily, bundle = load_training_bundle(
    OHLCV_PATH,
    benchmark_symbol=settings.universe.benchmark_symbol,
)

if SYMBOL_LIMIT:
    keep = sorted(hourly["symbol"].unique())[: int(SYMBOL_LIMIT)]
    hourly = hourly.loc[hourly["symbol"].isin(keep)].reset_index(drop=True)
    bundle.hourly = hourly
    bundle.daily = bundle.daily.loc[bundle.daily["symbol"].isin(keep)].reset_index(drop=True)

TRAIN_START = pd.Timestamp(hourly["ts"].min()).tz_convert("Asia/Kolkata").strftime("%Y-%m-%d")
TRAIN_END = pd.Timestamp(hourly["ts"].max()).tz_convert("Asia/Kolkata").strftime("%Y-%m-%d")

print("accelerator:", describe_accelerator())
print("gpu_available:", gpu_available())
print("parquet:", OHLCV_PATH)
print(f"hourly rows: {len(hourly)}, symbols: {hourly['symbol'].nunique()} ({TRAIN_START} to {TRAIN_END})")

accelerator: GPU: XGBoost CUDA + LightGBM GPU
gpu_available: True
parquet: /content/drive/MyDrive/ml_train/ml_scanner/train_ohlcv_60minute.parquet
hourly rows: 2976111, symbols: 401 (2021-09-06 to 2026-09-07)


## M03 — NSE calendar (7 hourly bars per session)



In [7]:
import pandas as pd
from ml_scan.data.calendar import NSECalendar

cal = NSECalendar()
n = len(cal.hourly_index(pd.Timestamp("2024-01-02", tz="Asia/Kolkata")))
print("hourly bars on 2024-01-02:", n)
assert n == 7


hourly bars on 2024-01-02: 7


## M04 — Parquet hourly reader (replaces TimescaleAdapter)

Expect standard OHLCV columns, `interval == 60minute`, and **no** 15:15 partial hours on the feature panel.
The 15:15 print is kept only for daily resampling.



In [8]:
assert len(hourly) > 0
assert {"ts", "symbol", "open", "high", "low", "close", "volume", "interval"} <= set(hourly.columns)
assert (hourly["interval"] == "60minute").all()
ts = pd.to_datetime(hourly["ts"], utc=True).dt.tz_convert("Asia/Kolkata")
assert not ts.dt.strftime("%H:%M").eq("15:15").any(), "15:15 bars must be dropped from the hourly feature panel"
probe = hourly.loc[hourly["symbol"] == hourly["symbol"].iloc[0]]
print(probe.head())
print("-" * 70)
print(probe.tail())


                         ts  symbol  instrument_token    open    high     low   close    volume source  n_5m  interval  is_partial_hour
0 2021-09-06 09:15:00+05:30  360ONE           3343617  396.25  404.75  396.25  400.50   20912.0   cagg    12  60minute            False
1 2021-09-06 10:15:00+05:30  360ONE           3343617  400.50  402.50  395.55  396.10   19828.0   cagg    12  60minute            False
2 2021-09-06 11:15:00+05:30  360ONE           3343617  396.10  397.50  396.05  397.50  362992.0   cagg    12  60minute            False
3 2021-09-06 12:15:00+05:30  360ONE           3343617  397.50  399.75  396.80  398.75   30628.0   cagg    12  60minute            False
4 2021-09-06 13:15:00+05:30  360ONE           3343617  398.75  398.75  398.30  398.50   25396.0   cagg    12  60minute            False
----------------------------------------------------------------------
                            ts  symbol  instrument_token    open    high     low   close    volume source  n_5m  

## M07 — Honest daily as-of join (daily resampled from hourly; no same-session leak)



In [9]:
from ml_scan.data.mtf_aligner import MTFAligner
from ml_scan.data.schemas import MTFBundle

syms = [s for s in ("RELIANCE", "TCS") if s in set(hourly["symbol"])]
if len(syms) < 2:
    syms = sorted(hourly["symbol"].unique())[:2]
h = hourly.loc[hourly["symbol"].isin(syms)].copy()
d = bundle.daily.loc[bundle.daily["symbol"].isin(syms)].copy()
aligned = MTFAligner().align_to_hourly(MTFBundle(hourly=h, daily=d, minute15=pd.DataFrame()))
print(aligned.groupby("symbol", sort=False).head(7)[["symbol", "ts", "d_ts", "d_close"]])
print("aligned", aligned.shape)
print("M07 ok — morning rows of the first session should have empty d_close (no same-session leak)")


        symbol                        ts                      d_ts  d_close
0     RELIANCE 2021-09-06 09:15:00+05:30                       NaT      NaN
1     RELIANCE 2021-09-06 10:15:00+05:30                       NaT      NaN
2     RELIANCE 2021-09-06 11:15:00+05:30                       NaT      NaN
3     RELIANCE 2021-09-06 12:15:00+05:30                       NaT      NaN
4     RELIANCE 2021-09-06 13:15:00+05:30                       NaT      NaN
5     RELIANCE 2021-09-06 14:15:00+05:30                       NaT      NaN
6     RELIANCE 2021-09-07 09:15:00+05:30 2021-09-06 00:00:00+05:30   1157.2
7423       TCS 2021-09-06 09:15:00+05:30                       NaT      NaN
7424       TCS 2021-09-06 10:15:00+05:30                       NaT      NaN
7425       TCS 2021-09-06 11:15:00+05:30                       NaT      NaN
7426       TCS 2021-09-06 12:15:00+05:30                       NaT      NaN
7427       TCS 2021-09-06 13:15:00+05:30                       NaT      NaN
7428       T

## M08 — TA engine parity (offline fixture, no warehouse)



In [10]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pytest", "tests/test_ta_engine_parity.py", "tests/test_parquet_source.py", "-q"])


0

## M09 — Hourly pandas_ta panel from the Drive parquet



In [11]:
from ml_scan.features.panel import PanelFeatureEngineer

feat_hourly_path = ARTIFACTS / "feat_hourly.parquet"
engineer = PanelFeatureEngineer(mode=settings.features.hourly_mode, atr_period=settings.features.atr_period)
print(f"building hourly features for {hourly['symbol'].nunique()} symbols, mode={settings.features.hourly_mode!r} …")
feat = engineer.transform(hourly).reset_index(drop=True)
feat.to_parquet(feat_hourly_path, index=False)
assert "ATR" in feat.columns
meta = {"symbol", "ts", "interval", "source", "instrument_token", "open", "high", "low", "close", "volume"}
n_ta = len([c for c in feat.columns if c not in meta])
print(f"hourly panel: {len(feat)} rows × {feat.shape[1]} columns ({n_ta} TA/derived)")
print("wrote", feat_hourly_path)
print(feat[["symbol", "ts"]].head())


building hourly features for 401 symbols, mode='full' …
hourly panel: 2976111 rows × 239 columns (229 TA/derived)
wrote /content/drive/MyDrive/ml_train/ml_scanner/artifacts/feat_hourly.parquet
   symbol                        ts
0  360ONE 2021-09-06 09:15:00+05:30
1  360ONE 2021-09-06 10:15:00+05:30
2  360ONE 2021-09-06 11:15:00+05:30
3  360ONE 2021-09-06 12:15:00+05:30
4  360ONE 2021-09-06 13:15:00+05:30


### M09b — Which stocks feed the model

The split later is **time-based**, not stock-based. Every symbol in the parquet is used on both train and test sides of each fold.



In [12]:
from IPython.display import display
from ml_scan.features.qc import symbol_row_counts
from ml_scan.features.ta_engine import generate_all_ta_features, generate_lite_ta_features

feat = pd.read_parquet(ARTIFACTS / "feat_hourly.parquet")
print(f"Q1 — {feat['symbol'].nunique()} symbols in the hourly panel")
display(symbol_row_counts(feat)[["symbol", "n_rows"]])

meta = {"symbol", "ts", "interval", "source", "instrument_token", "open", "high", "low", "close", "volume"}
print(f"Q2 — hourly_mode={settings.features.hourly_mode!r}: {len([c for c in feat.columns if c not in meta])} non-OHLCV columns")
sample = (
    feat.loc[feat["symbol"] == feat["symbol"].iloc[0], ["ts", "open", "high", "low", "close", "volume"]]
    .sort_values("ts")
    .set_index("ts")
)
ohlcv_cols = {"open", "high", "low", "close", "volume"}
print("lite", len(set(generate_lite_ta_features(sample).columns) - ohlcv_cols))
print("full", len(set(generate_all_ta_features(sample).columns) - ohlcv_cols))


Q1 — 401 symbols in the hourly panel


,symbol,n_rows
0,360ONE,7423
1,3MINDIA,7423
2,AARTIIND,7423
3,AAVAS,7423
4,ABB,7423
...,...,...
396,ZENSARTECH,7423
397,ZENTEC,7423
398,ZFCVINDIA,7423
399,ZYDUSLIFE,7423


Q2 — hourly_mode='full': 229 non-OHLCV columns
lite 22
full 227


## M10 — Daily (+ optional 15m) join

Daily EMA/ATR/RS come from resampled hourly (or `train_ohlcv_day.parquet` if you uploaded it).
`d_rs` uses `NIFTY 50` only when that symbol is in the parquet; otherwise it is the stock's own return.



In [ ]:
from ml_scan.features.htf import join_htf_features

feat = pd.read_parquet(ARTIFACTS / "feat_hourly.parquet")
mtf = join_htf_features(feat, bundle, benchmark_daily=bench_daily, config=settings.features)
mtf_path = ARTIFACTS / "feat_mtf.parquet"
mtf.to_parquet(mtf_path, index=False)
d_cols = [c for c in mtf.columns if c.startswith("d_")]
m15_cols = [c for c in mtf.columns if c.startswith("m15_")]
assert d_cols, "daily columns missing"
print("daily", d_cols[:10])
print("m15", m15_cols[:10] if m15_cols else "(none — upload train_ohlcv_15minute.parquet to add them)")
print("wrote", mtf_path, mtf.shape)


## M11 — Swing labels



In [15]:
from ml_scan.features.target import SwingLabeler

mtf = pd.read_parquet(ARTIFACTS / "feat_hourly.parquet")
tgt = settings.target
labeled = SwingLabeler(
    atr_col=tgt.atr_col,
    sl_mult=tgt.sl_mult,
    tp_r=tgt.tp_r,
    max_sessions=tgt.max_sessions,
    stop_wins_same_bar=tgt.stop_wins_same_bar,
).transform(mtf)
labeled_path = ARTIFACTS / "labeled.parquet"
labeled.to_parquet(labeled_path, index=False)
print(labeled["y"].value_counts(dropna=False))
print(labeled["y_reason"].value_counts(dropna=False))
print("wrote", labeled_path)


y
0.0    1845635
1.0    1042013
NaN      88463
Name: count, dtype: int64
y_reason
sl_hit        1845635
tp_hit        1042013
unresolved      88463
Name: count, dtype: int64
wrote /content/drive/MyDrive/ml_train/ml_scanner/artifacts/labeled.parquet


### M11b — Class weights (`cwts`)



In [16]:
from ml_scan.ml_engine.metrics import class_weight_dict

lab = pd.read_parquet(ARTIFACTS / "labeled.parquet")
resolved = lab.loc[lab["y"].isin([0, 1]), "y"].astype(int)
counts = resolved.value_counts().sort_index()
print(counts.to_string())
print(f"positive rate: {resolved.mean():.4f}")
cw = class_weight_dict(resolved)
print("class weights", {k: round(v, 4) for k, v in cw.items()})
print("weighted 0", cw[0] * int(counts.get(0, 0)))
print("weighted 1", cw[1] * int(counts.get(1, 0)))


y
0    1845635
1    1042013
positive rate: 0.3609
class weights {0: 0.7823, 1: 1.3856}
weighted 0 1443824.0
weighted 1 1443824.0


## M12 — Feature QC (drops price-level leakage suspects)



In [17]:
import json
from ml_scan.features.qc import run_qc

lab = pd.read_parquet(ARTIFACTS / "labeled.parquet")
_, qc_report = run_qc(
    lab,
    missing_threshold=settings.features.missing_threshold,
    out_path=ARTIFACTS / "feature_qc_report.json",
)
print(json.dumps({
    "n_rows": qc_report["n_rows"],
    "n_feature_cols": qc_report["n_feature_cols"],
    "n_leakage_suspects_excluded": len(qc_report["dropped_leakage_suspects"]),
    "leakage_ok": qc_report["leakage"]["ok"],
}, indent=2))


/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:135: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


{
  "n_rows": 2976111,
  "n_feature_cols": 158,
  "n_leakage_suspects_excluded": 58,
  "leakage_ok": false
}


### M12b — Baseline model bake-off (RF / XGBoost / LightGBM)

Same purged walk-forward folds. On a Colab GPU, **XGBoost uses CUDA**; pip LightGBM is usually CPU-only.
`SAMPLE_ROWS` keeps this stage tractable; the final train cell uses the full labeled panel.



In [18]:
import json
from IPython.display import display

from ml_scan.features.qc import select_xy
from ml_scan.ml_engine.estimator import compare_models, select_best_model
from ml_scan.ml_engine.metrics import fold_metrics_table
from ml_scan.ml_engine.splitter import PurgedWalkForward
from ml_scan.ops import stratified_sample

labeled = pd.read_parquet(ARTIFACTS / "labeled.parquet")
qc_report = json.loads((ARTIFACTS / "feature_qc_report.json").read_text(encoding="utf-8"))
initial_features = qc_report["features"]
print(f"{len(initial_features)} QC-passed features")

work = stratified_sample(labeled, SAMPLE_ROWS, random_state=settings.ml.random_state)
X, y, panel = select_xy(work, initial_features)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
baseline_summary, baseline_models = compare_models(
    X, y, panel, splitter=splitter, random_state=settings.ml.random_state
)
best_model = select_best_model(baseline_summary, metric="roc_auc")
display(baseline_summary.round(4))
print("best_model", best_model)
display(fold_metrics_table(baseline_models[best_model].fold_metrics_).round(4))
(ARTIFACTS / "model_comparison_initial.json").write_text(
    json.dumps(
        {
            "best_model": best_model,
            "n_initial_features": len(initial_features),
            "initial_features": initial_features,
            "summary": baseline_summary.reset_index().to_dict(orient="records"),
        },
        indent=2,
    ),
    encoding="utf-8",
)


158 QC-passed features


,model_key,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,n_folds,n_test_total
model,,,,,,,,,
LightGBM,lightgbm,0.7760,0.7845,0.6458,0.8121,0.7179,0.8754,4,181126
XGBoost,xgboost,0.7738,0.7832,0.6427,0.8126,0.7160,0.8730,4,181126
RandomForest,rf,0.7299,0.7422,0.5891,0.7822,0.6701,0.8121,4,181126


best_model lightgbm


,fold,n_train,n_test,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,class_weight_0,class_weight_1,n
0,1,72060,50765,0.4191,0.7730,0.7737,0.7086,0.7785,0.7419,0.8655,0.7902,1.3614,50765
1,2,132484,42450,0.3948,0.7602,0.7705,0.6575,0.8195,0.7296,0.8627,0.8075,1.3130,42450
2,3,186310,37107,0.3038,0.7864,0.7953,0.6108,0.8182,0.6995,0.8871,0.8141,1.2959,37107
3,4,230472,50804,0.2975,0.7846,0.7992,0.5989,0.8352,0.6976,0.8873,0.7999,1.3337,50804


3876

In [19]:
!nvidia-smi

Tue Sep  8 04:12:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             47W /  400W |     510MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## M13b — Boruta with the winning estimator



In [20]:
import json
from IPython.display import display

from ml_scan.features.qc import select_xy
from ml_scan.ml_engine.estimator import compare_models
from ml_scan.ml_engine.metrics import fold_metrics_table
from ml_scan.ml_engine.selector import BorutaSelector
from ml_scan.ml_engine.splitter import PurgedWalkForward
from ml_scan.ops import stratified_sample

labeled = pd.read_parquet(ARTIFACTS / "labeled.parquet")
init = json.loads((ARTIFACTS / "model_comparison_initial.json").read_text(encoding="utf-8"))
best_model, initial_features = init["best_model"], init["initial_features"]
work = stratified_sample(labeled, SAMPLE_ROWS, random_state=settings.ml.random_state)
X, y, _ = select_xy(work, initial_features)
boruta = BorutaSelector(
    max_iter=settings.ml.boruta_max_iter,
    random_state=settings.ml.random_state,
    estimator_name=best_model,
).fit(X, y)
boruta.save(ARTIFACTS / "boruta_features.json")
print(f"Boruta ({best_model}) kept {len(boruta.features_)} of {len(initial_features)}")
print(boruta.features_)

Xb, yb, panelb = select_xy(work, boruta.features_)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
boruta_summary, boruta_models = compare_models(
    Xb, yb, panelb, models=(best_model,), splitter=splitter, random_state=settings.ml.random_state
)
display(boruta_summary.round(4))
display(fold_metrics_table(boruta_models[best_model].fold_metrics_).round(4))
(ARTIFACTS / "model_comparison_boruta.json").write_text(
    json.dumps(
        {"n_features": len(boruta.features_), "summary": boruta_summary.reset_index().to_dict(orient="records")},
        indent=2,
    ),
    encoding="utf-8",
)


Boruta (lightgbm) kept 58 of 158
['REFLEX', 'CCI', 'CFO', 'CG', 'CRSI', 'CTI', 'BULLP_13', 'BEARP_13', 'D_9_3', 'J_9_3', 'KSTs_9', 'MACDh_12_26_9', 'MOM', 'PGO', 'PPO_12_26_9', 'PPOh_12_26_9', 'SLOPE', 'SMIo_5_20_5_1.0', 'STOCHFk_14_3', 'TRIX_30_9', 'TRIXs_30_9', 'TSIs_13_25_13', 'UO', 'WILLR', 'LOG_RETURN', 'PERCENT_RETURN', 'MAD', 'TOS_STDEVALL_LR', 'TOS_STDEVALL_L_1', 'TOS_STDEVALL_L_2', 'TOS_STDEVALL_L_3', 'TOS_STDEVALL_U_3', 'ZSCORE', 'DMP_14', 'AROOND_14', 'DPO', 'RWIh_14', 'VTXP_14', 'ABER_ATR_5_15', 'ATR', 'BBB_5_2.0_2.0', 'BBP_5_2.0_2.0', 'HWW_1', 'HWPCT_1', 'MASSI', 'NATR', 'THERMOma_20_2_0.5', 'TRUE_RANGE', 'UI', 'AD', 'CMF', 'EOM', 'NVI', 'PVIe_255', 'PVOh_12_26_9', 'PVOL', 'PVT', 'VHM']


,model_key,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,n_folds,n_test_total
model,,,,,,,,,
LightGBM,lightgbm,0.776,0.7849,0.6456,0.8137,0.7183,0.8758,4,181126


,fold,n_train,n_test,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,class_weight_0,class_weight_1,n
0,1,72060,50765,0.4191,0.7727,0.7736,0.7079,0.7791,0.7418,0.8658,0.7902,1.3614,50765
1,2,132484,42450,0.3948,0.7612,0.7717,0.6582,0.8218,0.7310,0.8632,0.8075,1.3130,42450
2,3,186310,37107,0.3038,0.7855,0.7949,0.6094,0.8188,0.6987,0.8874,0.8141,1.2959,37107
3,4,230472,50804,0.2975,0.7850,0.8001,0.5992,0.8377,0.6986,0.8878,0.7999,1.3337,50804


388

## M14b — VIF prune + same-model comparison



In [21]:
import json
from IPython.display import display

from ml_scan.features.qc import select_xy
from ml_scan.ml_engine.estimator import compare_models
from ml_scan.ml_engine.metrics import fold_metrics_table
from ml_scan.ml_engine.selector import VIFPruner, load_feature_list
from ml_scan.ml_engine.splitter import PurgedWalkForward
from ml_scan.ops import stratified_sample

labeled = pd.read_parquet(ARTIFACTS / "labeled.parquet")
init = json.loads((ARTIFACTS / "model_comparison_initial.json").read_text(encoding="utf-8"))
best_model = init["best_model"]
boruta_feats = load_feature_list(ARTIFACTS / "boruta_features.json")
work = stratified_sample(labeled, SAMPLE_ROWS, random_state=settings.ml.random_state)
X, _y, _ = select_xy(work, boruta_feats)
pruner = VIFPruner(max_vif=settings.ml.max_vif).fit(X)
pruner.save(ARTIFACTS / "selected_features.json")
print(f"VIF kept {len(pruner.features_)} of {len(boruta_feats)}")

Xs, ys, panels = select_xy(work, pruner.features_)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
vif_summary, vif_models = compare_models(
    Xs, ys, panels, models=(best_model,), splitter=splitter, random_state=settings.ml.random_state
)
display(vif_summary.round(4))
display(fold_metrics_table(vif_models[best_model].fold_metrics_).round(4))
(ARTIFACTS / "model_comparison_vif.json").write_text(
    json.dumps({"n_features": len(pruner.features_), "summary": vif_summary.reset_index().to_dict(orient="records")}, indent=2),
    encoding="utf-8",
)


VIF kept 32 of 58


,model_key,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,n_folds,n_test_total
model,,,,,,,,,
LightGBM,lightgbm,0.7769,0.7839,0.6492,0.8049,0.7169,0.8726,4,181126


,fold,n_train,n_test,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,class_weight_0,class_weight_1,n
0,1,72060,50765,0.4191,0.7716,0.7710,0.7106,0.7675,0.7380,0.8613,0.7902,1.3614,50765
1,2,132484,42450,0.3948,0.7649,0.7727,0.6666,0.8096,0.7312,0.8614,0.8075,1.3130,42450
2,3,186310,37107,0.3038,0.7877,0.7949,0.6137,0.8132,0.6995,0.8851,0.8141,1.2959,37107
3,4,230472,50804,0.2975,0.7844,0.7982,0.5991,0.8324,0.6967,0.8839,0.7999,1.3337,50804


390

## M15 — Purged walk-forward (no timestamp overlap)



In [22]:
import subprocess
import sys
from IPython.display import display

from ml_scan.features.qc import select_xy, symbol_row_counts
from ml_scan.ml_engine.selector import load_feature_list
from ml_scan.ml_engine.splitter import PurgedWalkForward, fold_symbol_table

subprocess.check_call([sys.executable, "-m", "pytest", "tests/test_purged_split.py", "-q"])

labeled = pd.read_parquet(ARTIFACTS / "labeled.parquet")
selected = load_feature_list(ARTIFACTS / "selected_features.json")
_, _, panel = select_xy(labeled, selected)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
print("resolved-label rows per symbol:")
display(symbol_row_counts(labeled).head(20))
fold_table = fold_symbol_table(panel, splitter)
print(f"{fold_table['symbol'].nunique()} symbols across {fold_table['fold'].nunique()} folds")
display(
    fold_table.pivot_table(index="symbol", columns="fold", values=["n_train_rows", "n_test_rows"], aggfunc="sum")
    .fillna(0)
    .astype(int)
    .head(15)
)


resolved-label rows per symbol:


,symbol,n_rows,n_win,n_loss,n_unresolved,win_rate
0,360ONE,7423,2768,4364,291,0.388110
1,3MINDIA,7423,2211,4911,301,0.310447
2,AARTIIND,7423,2399,4829,195,0.331904
3,AAVAS,7423,2299,4837,287,0.322169
4,ABB,7423,2745,4440,238,0.382046
5,ABBOTINDIA,7423,2262,4882,279,0.316629
6,ABCAPITAL,7423,2703,4516,204,0.374429
7,ABFRL,7423,2279,4955,189,0.315040
8,ABREL,7423,2642,4598,183,0.364917
9,ACC,7423,2233,5001,189,0.308681


401 symbols across 4 folds


n_test_rows                   n_train_rows                  
fold                 1     2     3     4            1     2     3     4
symbol                                                                 
360ONE            1296  1394  1388  1389         1385  2751  4215  5673
3MINDIA           1365  1359  1360  1343         1425  2860  4289  5709
AARTIIND          1358  1382  1370  1367         1479  2907  4359  5799
AAVAS             1354  1406  1314  1357         1427  2851  4325  5709
ABB               1361  1382  1380  1354         1428  2859  4311  5761
ABBOTINDIA        1359  1392  1389  1303         1421  2850  4312  5771
ABCAPITAL         1389  1364  1372  1364         1458  2917  4345  5787
ABFRL             1332  1373  1396  1384         1474  2876  4317  5783
ABREL             1376  1400  1394  1376         1415  2861  4331  5794
ACC               1366  1378  1390  1365         1462  2898  4343  5799
ACE               1369  1383  1391  1370         1443  2882  4334  5795
ACUTAAS           1357  1355  1362  1357         1374  2801  4222  5654
ADANIENSOL        1345  1352  1360  1342         1427  2842  4263  5693
ADANIENT          1346  1324  1378  1352         1462  2878  4272  5720
ADANIGREEN        1364  1345  1397  1334         1427  2861  4274  5741

## M16a — RandomizedSearch on purged folds (sampled)



In [23]:
import json
from IPython.display import display

from ml_scan.features.qc import select_xy
from ml_scan.ml_engine.optimize import random_search
from ml_scan.ml_engine.selector import load_feature_list
from ml_scan.ml_engine.splitter import PurgedWalkForward
from ml_scan.ops import stratified_sample

labeled = pd.read_parquet(ARTIFACTS / "labeled.parquet")
best_model = json.loads((ARTIFACTS / "model_comparison_initial.json").read_text(encoding="utf-8"))["best_model"]
selected = load_feature_list(ARTIFACTS / "selected_features.json")
work = stratified_sample(labeled, SAMPLE_ROWS, random_state=settings.ml.random_state)
Xs, ys, panels = select_xy(work, selected)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
best_params, search_results = random_search(
    Xs, ys, panels, model_name=best_model, splitter=splitter, n_iter=20, random_state=settings.ml.random_state
)
print("best_model", best_model)
print(json.dumps(best_params, indent=2, default=str))
(ARTIFACTS / "best_params.json").write_text(
    json.dumps({"model_name": best_model, "best_params": best_params}, indent=2, default=str),
    encoding="utf-8",
)
display(search_results.head(10))


best_model lightgbm
{
  "subsample": 0.8,
  "reg_lambda": 1.0,
  "num_leaves": 63,
  "n_estimators": 400,
  "min_child_samples": 20,
  "max_depth": 6,
  "learning_rate": 0.05,
  "colsample_bytree": 0.7
}


,param_subsample,param_reg_lambda,param_num_leaves,param_n_estimators,param_min_child_samples,param_max_depth,param_learning_rate,param_colsample_bytree,mean_test_score,std_test_score,rank_test_score
0,0.8,1.0,63,400,20,6,0.05,0.7,0.874766,0.012382,1
1,1.0,0.1,31,400,5,5,0.05,0.8,0.874316,0.012101,2
2,0.7,5.0,31,200,20,5,0.08,0.8,0.874034,0.011796,3
3,1.0,0.0,15,400,10,4,0.08,0.6,0.873905,0.012071,4
4,1.0,0.1,63,300,20,4,0.08,0.6,0.873479,0.011711,5
5,0.8,0.1,31,300,5,5,0.03,0.8,0.872559,0.011544,6
6,0.8,1.0,63,200,5,4,0.08,1.0,0.871967,0.011724,7
7,1.0,0.0,31,400,40,3,0.10,0.7,0.871582,0.011811,8
8,0.6,0.1,31,100,20,5,0.08,0.8,0.871568,0.011451,9
9,0.8,0.0,15,150,40,5,0.08,0.7,0.871514,0.011510,10


## M16b — Final train on the **full** labeled panel (GPU if available)

This is the model you copy to the other project. Walk-forward fold metrics are reported; the last fold's fitted trees are saved.



In [24]:
import json
from IPython.display import display

from ml_scan.features.qc import select_xy
from ml_scan.ml_engine.artifacts import save_model
from ml_scan.ml_engine.estimator import MLEstimator
from ml_scan.ml_engine.metrics import fold_metrics_table
from ml_scan.ml_engine.selector import load_feature_list
from ml_scan.ml_engine.splitter import PurgedWalkForward

labeled = pd.read_parquet(ARTIFACTS / "labeled.parquet")
best_model = json.loads((ARTIFACTS / "model_comparison_initial.json").read_text(encoding="utf-8"))["best_model"]
best_params = json.loads((ARTIFACTS / "best_params.json").read_text(encoding="utf-8"))["best_params"]
selected = load_feature_list(ARTIFACTS / "selected_features.json")
Xs, ys, panels = select_xy(labeled, selected)
splitter = PurgedWalkForward(n_splits=settings.ml.n_splits, embargo_sessions=settings.ml.embargo_sessions)
tuned = MLEstimator(
    model=best_model,
    random_state=settings.ml.random_state,
    params=best_params,
    use_class_weight=settings.ml.use_class_weight,
)
tuned.fit_walk_forward(Xs, ys, panels, splitter)
display(fold_metrics_table(tuned.fold_metrics_).round(4))
print(json.dumps(tuned.aggregate_metrics(), indent=2))
model_path = save_model(tuned, ARTIFACTS / "model.joblib")
print("wrote", model_path)
print("sidecar", model_path.with_suffix(".json"))


,fold,n_train,n_test,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,class_weight_0,class_weight_1,n
0,1,578944,546293,0.3996,0.7781,0.7809,0.6941,0.7949,0.7411,0.8717,0.7874,1.3699,546293
1,2,1153156,549685,0.4039,0.7805,0.7843,0.6984,0.8036,0.7473,0.8750,0.8031,1.3249,549685
2,3,1729566,555035,0.3286,0.7838,0.7935,0.6314,0.8217,0.7141,0.8841,0.8121,1.3009,555035
3,4,2312421,547372,0.3177,0.7938,0.8008,0.6360,0.8202,0.7165,0.8905,0.7945,1.3488,547372


{
  "accuracy": 0.7840378277690213,
  "balanced_accuracy": 0.7898725399799293,
  "precision": 0.6648997421526565,
  "recall": 0.8101636490561962,
  "f1": 0.7297163670945473,
  "roc_auc": 0.8803436098694899,
  "n_folds": 4,
  "n_test_total": 2198385
}
wrote /content/drive/MyDrive/ml_train/ml_scanner/artifacts/model.joblib
sidecar /content/drive/MyDrive/ml_train/ml_scanner/artifacts/model.json


## M17 — Scan latest bar from the labeled panel (no live warehouse)



In [7]:
from IPython.display import display
from ml_scan.execution.scanner import InferenceScanner
from pathlib import Path
import pandas as pd

# Ensure settings and bundle are available
if 'settings' not in globals() or 'bundle' not in globals():
    from ml_scan.config import load_settings
    from ml_scan.data.parquet_source import load_training_bundle
    settings = load_settings()
    REPO_DIR = Path("/content/drive/MyDrive/ml_train/ml_scanner")
    OHLCV_PATH = REPO_DIR / "train_ohlcv_60minute.parquet"
    _, _, bundle = load_training_bundle(
        OHLCV_PATH,
        benchmark_symbol=settings.universe.benchmark_symbol,
    )

# Convert the string to a Path object to allow the / operator
ARTIFACTS = Path("/content/drive/MyDrive/ml_train/ml_scanner/artifacts")

lab = pd.read_parquet(ARTIFACTS / "labeled.parquet")
scanner = InferenceScanner(settings)
scanner.load_artifacts(ARTIFACTS / "model.joblib", ARTIFACTS / "selected_features.json")
scan = scanner.run("latest", lab, daily=bundle.daily)
scan_path = ARTIFACTS / "scan_latest.csv"
scan.to_csv(scan_path, index=False)
print("wrote", scan_path)
display(scan)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


wrote /content/drive/MyDrive/ml_train/ml_scanner/artifacts/scan_latest.csv


,symbol,asof_ts,score,p_win,entry_px,sl_px,tp_px,atr,adtv_20,rank
0,APOLLOHOSP,2026-09-07 14:15:00+05:30,0.951349,0.951349,8825.00,8728.099242,9018.801515,48.450379,3.454029e+09,1
1,BOSCHLTD,2026-09-07 14:15:00+05:30,0.929540,0.929540,48450.00,47674.629420,50000.741160,387.685290,2.247638e+09,2
2,SUNDARMFIN,2026-09-07 14:15:00+05:30,0.858879,0.858879,4714.20,4639.436820,4863.726361,37.381590,2.104295e+08,3
3,LEMONTREE,2026-09-07 14:15:00+05:30,0.853491,0.853491,107.36,105.805732,110.468535,0.777134,3.188195e+08,4
4,JUBLPHARMA,2026-09-07 14:15:00+05:30,0.849438,0.849438,952.00,929.751852,996.496295,11.124074,3.507817e+08,5
...,...,...,...,...,...,...,...,...,...,...
396,KPITTECH,2026-09-07 14:15:00+05:30,0.019817,0.019817,560.50,551.642604,578.214792,4.428698,8.501451e+08,397
397,INFY,2026-09-07 14:15:00+05:30,0.019220,0.019220,1087.40,1071.552898,1119.094205,7.923551,8.748494e+09,398
398,KARURVYSYA,2026-09-07 14:15:00+05:30,0.019138,0.019138,342.00,335.920364,354.159272,3.039818,5.501933e+08,399
399,ICICIPRULI,2026-09-07 14:15:00+05:30,0.014553,0.014553,474.95,466.771987,491.306025,4.089006,6.071690e+08,400


## M18 / M19 — Cost model and next-open fill (offline tests)



In [12]:
import subprocess
import sys
import os
from pathlib import Path

# Ensure we are in the project root
if 'REPO_DIR' in globals():
    os.chdir(REPO_DIR)

# Diagnostic check for file existence
test_files = ["tests/test_costs.py", "tests/test_fill_lag.py"]
missing = [f for f in test_files if not Path(f).exists()]

if missing:
    print(f"Error: The following test files were not found: {missing}")
    print(f"Current Working Directory: {Path.cwd()}")
else:
    try:
        subprocess.check_call([sys.executable, "-m", "pytest", "tests/test_costs.py", "tests/test_fill_lag.py", "-q"])
        print("Tests passed successfully.")
    except subprocess.CalledProcessError as e:
        print(f"Pytest failed with return code {e.returncode}")

Tests passed successfully.


## M20–M22 — History signals, backtest on parquet hourly bars, report



In [9]:
import json
from IPython.display import display

from ml_scan.backtest.engine import Backtester
from ml_scan.backtest.metrics import compute_metrics, equity_to_frame
from ml_scan.execution.scanner import InferenceScanner, history_signals
from ml_scan.reporting.charts import equity_figure, importance_figure, write_html
from ml_scan.ml_engine.artifacts import load_model

lab = pd.read_parquet(ARTIFACTS / "labeled.parquet")
scanner = InferenceScanner(settings)
scanner.load_artifacts(ARTIFACTS / "model.joblib", ARTIFACTS / "selected_features.json")
hist = history_signals(scanner.score_panel(lab), threshold=settings.ml.score_threshold)
hist_path = ARTIFACTS / "scan_history.parquet"
hist.to_parquet(hist_path, index=False)
print("history signals", len(hist))

bt_dir = ARTIFACTS / "bt"
bt_dir.mkdir(parents=True, exist_ok=True)
result = Backtester(settings).run(hist, bundle)
result.metrics = compute_metrics(
    result.equity if result.equity is not None else pd.Series(dtype=float),
    result.trades,
    rf_annual=settings.backtest.rf_annual,
    periods=settings.backtest.trading_days_per_year,
    starting_equity=settings.risk.starting_equity,
)
if result.blotter is not None:
    result.blotter.to_parquet(bt_dir / "trades.parquet", index=False)
if result.equity is not None:
    equity_to_frame(result.equity).to_parquet(bt_dir / "equity.parquet", index=False)
(bt_dir / "metrics.json").write_text(json.dumps(result.metrics, indent=2), encoding="utf-8")
print(json.dumps(result.metrics, indent=2))

eq = pd.read_parquet(bt_dir / "equity.parquet")
fig = equity_figure(eq)
fig.show()
write_html(fig, ARTIFACTS / "report.html")
imps = load_model(ARTIFACTS / "model.joblib").feature_importances()
if not imps.empty:
    importance_figure(imps).show()
print("wrote", ARTIFACTS / "report.html")


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


history signals 1212533
{
  "cagr": 2.1456600484993693,
  "sharpe": 6.711604881887444,
  "sortino": 12.061503808861142,
  "max_dd": -0.08190750776406086,
  "win_rate": 0.6630109670987039,
  "profit_factor": 3.235488032938218,
  "n_trades": 2006.0,
  "avg_r": 1.1270388892618834,
  "exposure": 0.0
}


wrote /content/drive/MyDrive/ml_train/ml_scanner/artifacts/report.html


## Package — `ml_signal_model_ddMMMyyyy`

Writes the joblib + JSON sidecar into `MyDrive/ml_train` (and `artifacts/`) for the other project.
The other project needs: the `.joblib` file, `selected_features.json`, and the same hourly feature recipe (`PanelFeatureEngineer` + daily join). No scaler.



In [13]:
import json
import shutil
from datetime import date
from pathlib import Path

# Ensure DRIVE_DIR and ARTIFACTS are available
if 'DRIVE_DIR' not in globals():
    DRIVE_DIR = Path("/content/drive/MyDrive/ml_train/ml_scanner")
if 'ARTIFACTS' not in globals():
    ARTIFACTS = DRIVE_DIR / "artifacts"

stamp = date.today().strftime("%d%b%Y")  # e.g. 08Sep2026
stem = f"ml_signal_model_{stamp}"
src_joblib = ARTIFACTS / "model.joblib"
src_json = ARTIFACTS / "model.json"
src_feats = ARTIFACTS / "selected_features.json"

assert src_joblib.is_file(), "run M16b first"

for dest_dir in (DRIVE_DIR, ARTIFACTS):
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_joblib, dest_dir / f"{stem}.joblib")
    shutil.copy2(src_json, dest_dir / f"{stem}.json")
    shutil.copy2(src_feats, dest_dir / f"{stem}_features.json")

meta = json.loads(src_json.read_text(encoding="utf-8"))
print("packaged", stem)
print("model", meta.get("model_name"), "features", len(meta.get("features") or []))
print("aggregate", json.dumps(meta.get("aggregate_metrics"), indent=2))
print("Drive copies:")
for p in sorted(DRIVE_DIR.glob(f"{stem}*")):
    print(" ", p, p.stat().st_size)

packaged ml_signal_model_08Sep2026
model lightgbm features 32
aggregate {
  "accuracy": 0.7840378277690213,
  "balanced_accuracy": 0.7898725399799293,
  "precision": 0.6648997421526565,
  "recall": 0.8101636490561962,
  "f1": 0.7297163670945473,
  "roc_auc": 0.8803436098694899,
  "n_folds": 4,
  "n_test_total": 2198385
}
Drive copies:
  /content/drive/MyDrive/ml_train/ml_scanner/ml_signal_model_08Sep2026.joblib 2808808
  /content/drive/MyDrive/ml_train/ml_scanner/ml_signal_model_08Sep2026.json 2967
  /content/drive/MyDrive/ml_train/ml_scanner/ml_signal_model_08Sep2026_features.json 2788
